<a href="https://colab.research.google.com/github/zackykurniawan/data-science-2026/blob/main/Pertemuan4_MuhammadZackyKurniawan_240401010217.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_iris

## Langkah 1 — Load & Inspect Dataset

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
df = df.drop('target', axis=1)

print('Shape:', df.shape)
print(df.dtypes)
print(df.head())
print(df.describe().round(3))

## Langkah 2 — Statistik Deskriptif Lengkap

In [ ]:
for col_name in df.select_dtypes(include='number').columns:
    col = df[col_name]
    print(f'\n=== {col_name} ===')
    print(f'  Mean      : {col.mean():.3f}')
    print(f'  Median    : {col.median():.3f}')
    print(f'  Std Dev   : {col.std():.3f}')
    print(f'  Skewness  : {col.skew():.3f}')
    print(f'  Kurtosis  : {col.kurt():.3f}')

## Langkah 3 — Analisis Distribusi (Histogram + KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['sepal_length'], kde=True, color='#028090', bins=20, ax=ax)
ax.axvline(df['sepal_length'].mean(), color='red', linestyle='--',
           label=f"Mean={df['sepal_length'].mean():.2f}")
ax.axvline(df['sepal_length'].median(), color='orange', linestyle='--',
           label=f"Median={df['sepal_length'].median():.2f}")
ax.set_title('Distribusi Sepal Length')
ax.legend()
plt.show()

## Langkah 4 — Boxplot per Spesies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, x='species', y='petal_length',
            hue='species', palette='Set2', legend=False, ax=axes[0])
axes[0].set_title('Boxplot Petal Length')
sns.violinplot(data=df, x='species', y='petal_length',
               hue='species', palette='Set2', inner='box', legend=False, ax=axes[1])
axes[1].set_title('Violin Plot Petal Length')
plt.tight_layout()
plt.show()

## Langkah 5 — Matriks Korelasi Pearson

In [ ]:
corr = df.drop('species', axis=1).corr(method='pearson')
print(corr.round(3))

mask = np.triu(np.ones(corr.shape, dtype=bool))
corr_masked = corr.where(~mask)
max_pair = corr_masked.stack().idxmax()
print(f'Korelasi tertinggi: {max_pair} = {corr.loc[max_pair]:.3f}')

## Langkah 6 — Scatter Plot & Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='petal_length', y='petal_width',
                hue='species', palette='Set2', ax=axes[0])
m, b, *_ = stats.linregress(df['petal_length'], df['petal_width'])
xr = np.linspace(df['petal_length'].min(), df['petal_length'].max(), 100)
axes[0].plot(xr, m * xr + b, color='gray', linewidth=2)
axes[0].set_title('Scatter: Petal Length vs Petal Width')

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title('Heatmap Korelasi')

plt.tight_layout()
plt.show()

### Interpretasi Langkah 6

**Scatter Plot — Petal Length vs Petal Width:**
- Terdapat hubungan **linear positif yang sangat kuat** antara `petal_length` dan `petal_width` dengan nilai korelasi Pearson **r = 0.963**. Artinya, semakin panjang mahkota bunga, semakin lebar pula mahkota tersebut.
- Ketiga spesies membentuk **kluster yang terpisah jelas**: *setosa* berada di kiri bawah (petal kecil), *versicolor* di tengah, dan *virginica* di kanan atas (petal terbesar).
- Garis regresi linear berwarna abu-abu mengkonfirmasi tren positif yang konsisten di seluruh data.

**Heatmap Korelasi Pearson:**
- **Warna merah tua** pada pasangan `petal_length` & `petal_width` (r = 0.96) dan `sepal_length` & `petal_length` (r = 0.87) menunjukkan korelasi positif sangat kuat.
- **Warna biru muda** pada pasangan yang melibatkan `sepal_width` menunjukkan korelasi negatif lemah, artinya lebar kelopak hampir tidak berhubungan dengan variabel lain.
- Diagonal bernilai 1.00 (merah penuh) karena setiap variabel berkorelasi sempurna dengan dirinya sendiri.

> ⚠️ **Catatan penting:** Korelasi tinggi **tidak berarti kausalitas**. Korelasi kuat antara `petal_length` dan `petal_width` kemungkinan besar dipengaruhi oleh variabel ketiga, yaitu **spesies** (*confounding variable*) — terbukti dari kluster yang terpisah pada scatter plot.